目的/Objective
* KerasでCNNを構築しMNISTの画像分類を行う。
* Build a CNN with Keras and perform image classification on MNIST.
* https://keras.io/examples/vision/mnist_convnet/
* https://www.tensorflow.org/tutorials/images/intro_to_cnns?hl=ja

データセット/Dataset
* 今回の文字認識ではMNIST Datasetを用いる。[MNIST Dataset](http://yann.lecun.com/exdb/mnist/)は0から9までの数字が記述されている画像から構成されたデータセットである。MNIST Datasetの文字画像は以下のように白黒で比較的認識しやすいように画像処理されている。
* In this character recognition task, the MNIST dataset is used. The [MNIST Dataset](http://yann.lecun.com/exdb/mnist/) is a dataset composed of images containing handwritten digits from 0 to 9. The digit images in the MNIST dataset are processed in grayscale (black and white) so that they are relatively easy to recognize.

* 0〜9の手書き数字の画像から構成/Composed of images of handwritten digits from 0 to 9.
* 画像サイズ/Image size：28×28 pixels
* 学習用画像/Training image：60,000
* 評価用画像/Evaluation image：10,000

![MNIST_sample.png](https://qiita-image-store.s3.ap-northeast-1.amazonaws.com/0/143078/559938dc-9a99-d426-010b-e000bca0aac6.png)

深層学習における基本的実装手順/Basic implementation procedure in deep learning
1. データの読み込み/Loading data
2. 訓練データと評価データの準備/Data preparation for training and evaluation
3. アルゴリズムの選択/Algorithm selection
4. 学習/Training
5. モデルの評価/Model evaluation

準備/Preparation
* Graphics Processing Unit (GPU)を用いて処理を行うために、上部のメニューバーの「ランタイム」→「ランタイムのタイプを変更」からハードウェアアクセラレータをGPUにしてください。
* To perform processing using a Graphics Processing Unit (GPU), please go to the top menu bar and select "Runtime" → "Change runtime type", then set the hardware accelerator to GPU.

ライブラリとモジュールのインポート/Importing libraries and modules

In [1]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

データの準備/Data preparation
* MNIST のピクセルの情報は、0〜255 の数値で提供されており、0 が白、255に近くにつれて黒になるような数字になっています。 数値計算の都合上、入力を 0〜1の範囲の数値にした方が良いため、データ型を float に変換したのち、255 で割ります。
* The pixel values in MNIST are provided as numbers ranging from 0 to 255, where 0 represents white and values closer to 255 represent darker (black) pixels. For numerical computation purposes, it is better to scale the inputs to the range 0 to 1. Therefore, we first convert the data type to float and then divide by 255.

NumPy's expand_dims 関数/NumPy's expand_dims function
* 指定された位置に新しい軸を挿入することによって配列の次元を拡張するために使用されます。
* It is used to expand the dimensions of an array by inserting a new axis at a specified position.
* https://pystyle.info/numpy-modify-axis-and-shape/#index_id4
* https://runebook.dev/ja/docs/numpy/reference/generated/numpy.expand_dims
* numpy.expand_dims: https://numpy.org/doc/stable/reference/generated/numpy.expand_dims.html

正解データの加工
* MNIST の正解データは、対となる画像に何の数字が書かれているかを表す「0」〜「9」の数字を提供しています。 しかし、ここで作成する人工知能は前述の通り出力が10個の確率になりますので、今のままでは比較が難しいです。
* そこで「0」〜「9」までの数字の形を変換します。 どのように変換するのかといえば、例えば「4」という数字であれば [0,0,0,0,1,0,0,0,0,0] というような、「4」を表す値だけが「1」となるような1次元配列にします。
* keras.utils.to_categorical: ラベルデータの整数値を2値クラスの行列に変換（例「整数 1」を「0,1,0,0,0,0,0,0,0,0」と表現
* keras.utils.to_categorical: https://keras.io/ja/utils/
---
Processing of ground truth data
* The ground truth data for MNIST provides a digit from “0” to “9” indicating which number is written in the corresponding image. However, as mentioned above, the artificial intelligence model we are building outputs ten probabilities, so in its current form it is difficult to compare them directly.
* There, we convert the shapes of the digits “0” through “9.”
How we convert them is, for example, if the digit is “4,” we represent it as a one-dimensional array like [0,0,0,0,1,0,0,0,0,0], where only the value corresponding to “4” is set to 1.
* keras.utils.to_categorical: Converts integer label data into a binary class matrix (e.g., represents the integer "1" as "0,1,0,0,0,0,0,0,0,0").
* keras.utils.to_categorical: https://keras.io/ja/utils/

In [2]:
# データパラメータ/data parameters
num_classes = 10
input_shape = (28, 28, 1)

# 1. データの読み込み/Loading data
# 2. 訓練データと評価データの準備: 訓練セットとテストセットに分ける/Preparation of training and evaluation data: Split into a training set and a test set
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# 画像のピクセルの値を 0~1 の間に正規化/Normalize the pixel values of the image to the range 0–1
x_train = x_train.astype("float32") / 255 #整数から0～1間の浮動小数に変換/Convert an integer to a floating-point number between 0 and 1
x_test = x_test.astype("float32") / 255 #整数から0～1間の浮動小数に変換/Convert an integer to a floating-point number between 0 and 1
# 画像の形状を確認する/Check the shape of the image（28, 28, 1）
x_train = np.expand_dims(x_train, -1) #新たな次元を追加: 第一引数に元のndarray、第二引数に次元を追加する位置を指定/Add a new axis: first argument is the original ndarray, second argument is the axis position
x_test = np.expand_dims(x_test, -1) #新たな次元を追加: 第一引数に元のndarray、第二引数に次元を追加する位置を指定/Add a new axis: first argument is the original ndarray, second argument is the axis position
print("x_train shape:", x_train.shape)
print(x_train.shape[0], "train samples")
print(x_test.shape[0], "test samples")

# クラスベクトルを2値クラス行列に変換する/Convert class vectors to binary class matrices
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
x_train shape: (60000, 28, 28, 1)
60000 train samples
10000 test samples


モデルの構築 Sequentialモデル: https://www.tensorflow.org/guide/keras/sequential_model?hl=ja
* Sequential モデルは、ニューラルネットワークを構築するための方法の一つです。
* 画像分類の問題では、Sequential モデルを使って、画像を入力とし、それがどのクラスに属するかを出力することができます。
* Keras でモデルを作成するには2つの方法:①Sequential モデル（tf.keras.Sequential）で作成する方法 ②FunctionalAPI（tf.keras.Model）で作成する方法
* モデル作成の流れ: https://zenn.dev/nekoallergy/articles/keras-create-model01
---
Building a Sequential Model: https://www.tensorflow.org/guide/keras/sequential_model?hl=ja
* Sequential model is one of the methods for building neural networks.
* In an image classification problem, you can use a Sequential model to take an image as input and output the class to which it belongs.
* There are two ways to create a model in Keras: 1. Using the **Sequential model** (`tf.keras.Sequential`) 2. Using the **Functional API** (`tf.keras.Model`)
* Process of model creation: https://zenn.dev/nekoallergy/articles/keras-create-model01

In [3]:
# 3. アルゴリズムの選択/algorithm selection
model = keras.Sequential(
    [
        keras.Input(shape=input_shape),
        layers.Conv2D(32, kernel_size=(3, 3), activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Flatten(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax"),
    ]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        16,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 34,826 (136.04 KB)

 Trainable params: 34,826 (136.04 KB)

 Non-trainable params: 0 (0.00 B)

出力形状
* バッチサイズ、出力ユニットの個数
* 任意のバッチサイズである場合は、バッチサイズはNone
* https://www.tensorflow.org/tutorials/images/intro_to_cnns?hl=ja
---
Output Shape
* batch size, number of output units
* If the batch size is arbitrary, the batch size is None.
* https://www.tensorflow.org/tutorials/images/intro_to_cnns?hl=ja

モデルのコンパイル
* モデルの学習を始める前に、compileメソッドを用いどのような学習処理を行なうかを設定する必要があります。compileメソッドは3つの引数を取ります。生成したmodelインスタンスからcompileメソッドを呼びだします。
 * 最適化アルゴリズム: 引数として定義されている最適化手法の識別子を文字列として与える（rmspropやadagradなど）、もしくは Optimizerクラスのインスタンスを与えることができます。
 * 損失関数: モデルが最小化しようとする目的関数です。引数として定義されている損失関数の識別子を文字列として与える（categorical_crossentropyやmseなど）、もしくは目的関数を関数として与えることができます。
 * 評価関数のリスト: 分類問題では精度としてmetrics=['accuracy']を指定したくなるでしょう。引数として定義されている評価関数の識別子を文字列として与える、もしくは自分で定義した関数を関数として与えることができます。
---
Compiling the model
* Before starting the training of a model, you need to use the compile method to set up what kind of training process will be performed. The compile method takes three arguments. You call the compile method from the generated model instance.
* Optimization algorithm: You can provide a string representing the identifier of the optimization method (such as 'rmsprop' or 'adagrad'), or an instance of the Optimizer class.
* Loss function: A function that the model tries to minimize. You can provide the loss function either as a string identifier (such as categorical_crossentropy or mse) or as a custom function.
* List of evaluation functions: For classification problems, you would typically want to specify metrics=['accuracy']. You can provide the identifier of a predefined evaluation function as a string, or you can supply a custom function directly.

学習の実行
* 学習するには、生成したmodelインスタンスから**fit()メソッド**を呼びだします。
* model.fit(入力データ、正解ラベル、バッチサイズ、エポック数）
* バッチサイズを小さくすると消費メモリが少なくて済むが、うまく動かなくなる。
* エポックは学習の繰り返す数
---
Execution of Learning
* To train, call the **fit() method** on the generated model instance.
* model.fit(input_data, target_labels, batch_size, epochs)
* Reducing the batch size decreases memory usage, but it may cause the model to not work properly.
* An epoch is the number of times the learning process is repeated.

メソッド/method: **compile**, **fit**, **evaluate**: https://keras.io/api/models/model_training_apis/

In [4]:
# 4. 学習/training
batch_size = 128
epochs = 15

model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"]) #多クラス分類用の交差エントロピー（Categorical Cross Entropy）

model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_split=0.1)

Epoch 1/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.8925 - loss: 0.3566 - val_accuracy: 0.9760 - val_loss: 0.0810
Epoch 2/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.9649 - loss: 0.1144 - val_accuracy: 0.9830 - val_loss: 0.0592
Epoch 3/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9733 - loss: 0.0860 - val_accuracy: 0.9870 - val_loss: 0.0490
Epoch 4/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9778 - loss: 0.0709 - val_accuracy: 0.9892 - val_loss: 0.0412
Epoch 5/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9811 - loss: 0.0631 - val_accuracy: 0.9888 - val_loss: 0.0398
Epoch 6/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9824 - loss: 0.0571 - val_accuracy: 0.9905 - val_loss: 0.0349
Epoch 7/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9836 - loss: 0.0515 - val_accuracy: 0.9905 - val_loss: 0.0357
Epoch 8/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9849 - loss: 0.0491 - val_accuracy: 0

* モデルを評価するには、生成したmodelインスタンスから**evaluateメソッド**を呼びだします。
* To evaluate a model, call the **evaluate method** on the generated model instance.

In [5]:
# 5. モデルの評価/Model evaluation
score = model.evaluate(x_test, y_test, verbose=0)
print("Test loss:", score[0]) #score[0]にlossを、score[1]にaccuracyを持つリストを返す/Return a list where score[0] contains the loss and score[1] contains the accuracy
print("Test accuracy:", score[1])

Test loss: 0.02508089877665043
Test accuracy: 0.9919999837875366
